# DeBERTa-v3-small (PRETRAINED TRANSFORMER)

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers==4.40.0', 'sentencepiece'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


CompletedProcess(args=['pip', 'install', '-q', 'transformers==4.40.0', 'sentencepiece'], returncode=0)

In [2]:
import os, warnings, pickle
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

INPUT_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
OUTPUT_DIR = Path('/kaggle/working')

Device : cuda
GPU    : Tesla T4


In [4]:
# creating logs and models folder in kaggle 

OUTPUT_DIR = Path('/kaggle/working')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "logs").mkdir(
    parents=True,
    exist_ok=True
)

(OUTPUT_DIR / "models").mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
# config define randomly chasing (taking best one)...

CFG = dict(
    model_name   = 'microsoft/deberta-v3-small',
    max_len      = 256,    # tokens for (prompt + ONE option)
    batch_size   = 16,    
    lr           = 2e-5,   # transformer learning rate
    weight_decay = 0.01,
    epochs       = 5,      # transformers converge fast
    warmup_ratio = 0.1,    # 10% of steps for warmup
    patience     = 3,
    seed         = 42,
)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print('Config:')
for k, v in CFG.items(): print(f'  {k:<14}: {v}')

Config:
  model_name    : microsoft/deberta-v3-small
  max_len       : 256
  batch_size    : 16
  lr            : 2e-05
  weight_decay  : 0.01
  epochs        : 5
  warmup_ratio  : 0.1
  patience      : 3
  seed          : 42


In [6]:
train_df = pd.read_csv(INPUT_DIR / 'train.csv')
test_df  = pd.read_csv(INPUT_DIR / 'test.csv')

# Lowercase it 
for col in ['prompt','A','B','C','D','E']:
    train_df[col] = train_df[col].str.lower().str.strip()
    test_df[col]  = test_df[col].str.lower().str.strip()

In [7]:
np.random.seed(CFG['seed'])
tr_idx, va_idx = [], []
for ans in 'ABCDE':
    idx = train_df[train_df['answer']==ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx)*0.8)
    tr_idx += idx[:cut]
    va_idx += idx[cut:]

tr_df = train_df.loc[tr_idx].reset_index(drop=True)
va_df = train_df.loc[va_idx].reset_index(drop=True)
te_df = test_df.copy()

In [8]:
A2I = {'A':0,'B':1,'C':2,'D':3,'E':4}
I2A = {v:k for k,v in A2I.items()}

print(f'Train:{len(tr_df)} | Val:{len(va_df)} | Test:{len(te_df)}')

Train:1599 | Val:401 | Test:500


In [9]:
# Convert each question into 5 (text, label) pairs.
# text  = prompt + single option
# label = 1 if this option is correct, else 0

def expand_pairs(df, is_test=False):
    rows = []
    for _, r in df.iterrows():
        correct = r.get('answer', None)
        for opt in 'ABCDE':
            rows.append({
                'id'    : r['id'],
                'option': opt,
                'text'  : f"{r['prompt']} {r[opt]}",
                'label' : int(opt == correct) if not is_test else -1
            })
    return pd.DataFrame(rows)

In [10]:
tr_pairs = expand_pairs(tr_df, is_test=False)
va_pairs = expand_pairs(va_df, is_test=False)
te_pairs = expand_pairs(te_df, is_test=True)

print(f'Train pairs: {len(tr_pairs):,}  ({len(tr_df)}×5)')
print(f'Val pairs  : {len(va_pairs):,}  ({len(va_df)}×5)')
print(f'Test pairs : {len(te_pairs):,}  ({len(te_df)}×5)')
print(f'\nLabel dist (train): {tr_pairs["label"].value_counts().to_dict()}')

Train pairs: 7,995  (1599×5)
Val pairs  : 2,005  (401×5)
Test pairs : 2,500  (500×5)

Label dist (train): {0: 6396, 1: 1599}


# LOAD TOKENIZER

In [11]:
print(f'Loading tokenizer: {CFG["model_name"]}')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

# Quick testing...
sample = tokenizer(
    'what is photosynthesis? process by which plants make food',
    max_length=CFG['max_len'],
    truncation=True,
    padding='max_length',
    return_tensors='pt'
)
print(f'Sample encoding shape: {sample["input_ids"].shape}')

Loading tokenizer: microsoft/deberta-v3-small


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer loaded! Vocab size: 128,000
Sample encoding shape: torch.Size([1, 256])


# Dataset

    Each item = one (prompt+option) pair tokenized for DeBERTa.
    Label = 1 if this option is the correct answer, else 0.

In [12]:
class OptionDataset(Dataset):
    def __init__(self, pairs_df, tokenizer, max_len, is_test=False):
        self.df       = pairs_df.reset_index(drop=True)
        self.tok      = tokenizer
        self.max_len  = max_len
        self.is_test  = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tok(
            row['text'],
            max_length    = self.max_len,
            truncation    = True,
            padding       = 'max_length',
            return_tensors= 'pt'
        )
        item = {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)

        if not self.is_test:
            item['label'] = torch.tensor(row['label'], dtype=torch.long)
        return item


nw = 2 if DEVICE.type == 'cuda' else 0
tr_ds = OptionDataset(tr_pairs, tokenizer, CFG['max_len'], is_test=False)
va_ds = OptionDataset(va_pairs, tokenizer, CFG['max_len'], is_test=False)
te_ds = OptionDataset(te_pairs, tokenizer, CFG['max_len'], is_test=True)

In [13]:
tr_loader = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=nw, pin_memory=True)
va_loader = DataLoader(va_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=nw, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=nw, pin_memory=True)

print(f'Train batches: {len(tr_loader)} | Val batches: {len(va_loader)}')
print('Datasets ready!')

Train batches: 500 | Val batches: 126
Datasets ready!


# DEBERTA MODEL

Approach:
    DeBERTa-v3-small with a binary classification head.
    Scores how likely each (prompt, option) pair is correct.

    Architecture:
      DeBERTa backbone (pretrained, 70M params)
          ↓  [CLS] token embedding  (768-dim)
      Dropout (0.3)
          ↓
      Linear (768 → 2)   [wrong, correct]

In [14]:
class DeBERTaOptionScorer(nn.Module):
    def __init__(self, model_name, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size   = self.backbone.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, 2)   # binary: wrong vs correct

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # DeBERTa outputs: last_hidden_state (B, L, 768)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            kwargs['token_type_ids'] = token_type_ids

        out = self.backbone(**kwargs)

        # Use [CLS] token (position 0) as sentence representation
        cls = out.last_hidden_state[:, 0, :]   # (B, 768)
        return self.fc(self.drop(cls))          # (B, 2)

In [15]:
model = DeBERTaOptionScorer(CFG['model_name']).to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded!')
print(f'Total params    : {total/1e6:.1f}M')
print(f'Trainable params: {trainable/1e6:.1f}M')

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Model loaded!
Total params    : 141.3M
Trainable params: 141.3M


# MAP3 Function

      Run model on all pairs → get P(correct) for each option → rank per question → compute MAP@3 and done .

In [16]:
def compute_map3(model, loader, pairs_df, question_df, device):
    model.eval()
    all_scores = []

    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ttids = batch.get('token_type_ids')
            if ttids is not None: ttids = ttids.to(device)

            logits = model(ids, mask, ttids)
            # P(correct) = softmax score for class 1
            scores = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_scores.extend(scores.tolist())

    # Attach scores to pairs
    tmp = pairs_df.copy()
    tmp['score'] = all_scores

    # Rank options per question and compute MAP@3
    map3_scores = []
    for _, qrow in question_df.iterrows():
        qid    = qrow['id']
        q_rows = tmp[tmp['id']==qid].sort_values('score', ascending=False)
        top3   = q_rows['option'].values[:3].tolist()
        true   = qrow['answer']
        score  = 1./(top3.index(true)+1) if true in top3 else 0.
        map3_scores.append(score)

    return float(np.mean(map3_scores))

In [17]:
def train_epoch(model, loader, optimizer, scheduler, scaler, device, criterion):
    model.train()
    total_loss = 0.
    all_preds, all_labels = [], []

    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        ttids = batch.get('token_type_ids')
        if ttids is not None: ttids = ttids.to(device)
        lbl   = batch['label'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            logits = model(ids, mask, ttids)
            loss   = criterion(logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(lbl.cpu().tolist())

    acc = sum(p==l for p,l in zip(all_preds,all_labels)) / len(all_labels)
    return total_loss/len(loader), acc

# WANDB SETUP

In [18]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=key)
    USE_WANDB = True
    print('WandB logged in!')
except:
    USE_WANDB = False
    print('WandB secret not found — continuing without it.')

if USE_WANDB:
    run = wandb.init(
        project='23f2003236-t22026',
        name='deberta_v3_small_v1',
        config=CFG,
        tags=['deberta','pretrained','transformer'],
        reinit=True
    )
    print(f'Run URL: {run.url}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB logged in!


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260708_065719-pd25p4jy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run deberta_v3_small_v1
wandb: ⭐️ View project at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/pd25p4jy


Run URL: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/pd25p4jy
